In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
from glob import glob
import numpy as np

files = sorted(glob('../output_results/E_sent/main/*.json'))
if not files:
    raise FileNotFoundError('No matching files found')

labels = {
    Path(file_path).name: f'tmp_{index + 1}'
    for index, file_path in enumerate(files)
}

file_labels = []
train_accuracies = []
test_accuracies = []
for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)

    file_name = Path(file_path).name
    file_labels.append('_'.join([data.get('training_config', {}).get('input_mode'), data.get('training_config', {}).get('output_file_prefix') or 'poisson_sc']))
    train_accuracies.append(data['results']['epoch_train_accuracy'][-1])
    test_accuracies.append(data['results']['test_accuracy'])

x = np.arange(len(files))
width = 0.4

fig, ax = plt.subplots(figsize=(max(8, len(files) * 1.2), 5))
ax.bar(x - width / 2, train_accuracies, width, label='Train accuracy', color='#4C78A8')
ax.bar(x + width / 2, test_accuracies, width, label='Test accuracy', color='#F58518')
ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.set_xticks(x)
ax.set_xticklabels(file_labels, rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

In [4]:
from E_sent_eval import evaluate_model
from glob import glob
from pathlib import Path
import json
from pprint import pprint
from argparse import Namespace

models = glob('../output_results/E_sent/main/*.pt')

for model_path in models:
    meta_path = Path(model_path).with_suffix('.json')
    meta = json.loads(meta_path.read_text()).get('training_config', {})
    args = Namespace(**(meta | {
        "model_path": model_path,
        "diagnose": False,
        # "diagnose": True,
        "limit": 1000,
        "split": "test",
        "estimate_energy": True,
        "energy_ac_cost_pj":25.63,
        "output_json": None,
    }))
    # try to cast each number-like argument to int if possible, since JSON doesn't distinguish
    args = Namespace(**{k: (int(v) if isinstance(v, str) and v.isdigit() else v) for k, v in vars(args).items()})
    print(model_path)
    # pprint(args)
    evaluate_model(args)

../output_results/E_sent/main\lat_sc_2026-04-28_13-42-47_e-50_s-25_spatial.pt
Evaluation | split=test | samples=1000 | loss=0.8105 | acc=0.7380
Average AC operations per sample: 156001.81
Average energy per sample: 3998326.23 pJ (3998.3262 nJ)
../output_results/E_sent/main\lat_ttfs_ce_2026-04-28_13-58-26_e-50_s-25_spatial.pt
Evaluation | split=test | samples=1000 | loss=0.7027 | acc=0.7230
TTFS fallback rate: 0.0460
TTFS mean first spike time (fired output neurons): 21.4282
Average AC operations per sample: 181584.32
Average energy per sample: 4654006.00 pJ (4654.0060 nJ)
../output_results/E_sent/main\sent_2026-04-28_10-13-09_e-50_s-25_spatial.pt
Evaluation | split=test | samples=1000 | loss=0.7398 | acc=0.7220
Average AC operations per sample: 1180890.06
Average energy per sample: 30266210.99 pJ (30266.2110 nJ)
../output_results/E_sent/main\sent_2026-04-28_10-30-42_e-50_s-25_temporal.pt
Evaluation | split=test | samples=1000 | loss=0.4781 | acc=0.7940
Average AC operations per sample:

TODO:
which model is most efficient/active by layer?
why has fastest inference time?
write up the sent analysis task methods